In [4]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.3 MB/s eta 0:00:00


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%run Main_Preprocessing_1.py

/content/Main_Preprocessing_1.py:9: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("genomic_features.csv")


Data loaded:
  df shape : (242035, 19)
  df2 shape: (698000, 9)
  df3 shape: (2266, 96)
  df4 shape: (2132, 49)

After MSI and Growth fill:
  MSI missing         : 630
  Growth Prop missing : 0

After tissue descriptor fills:
  GDSC Tissue descriptor 1 missing: 0
  GDSC Tissue descriptor 2 missing: 0

After feature column cleaning — missing counts:
TARGET                                     27872
DRUG_NAME                                      0
TCGA_DESC                                   1067
Microsatellite instability Status (MSI)      630
Growth Properties                              0
GDSC Tissue descriptor 1                       0
GDSC Tissue descriptor 2                       0

TCGA Stage 1 fill results:
  Missing before                             : 1067
  Filled from cancer-type column             : 360
  Filled manually                            : 707
  Filled from Tissue descriptor 2 (unique)  : 0
  Missing after                              : 0

TCGA Stage 2 (OTHER -> PRA

In [9]:
import optuna
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [10]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression")
print(f"R²:  {r2_lr:.4f}")
print(f"MAE: {mae_lr:.4f}")
print(f"RMSE:{rmse_lr:.4f}\n")

Linear Regression
R²:  0.7631
MAE: 1.0079
RMSE:1.3443



In [11]:
def objective_etr(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300, step=50)
    max_depth = trial.suggest_int('max_depth', 5, 50) or None
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5, 0.7])

    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                cv=KFold(3, shuffle=True, random_state=42),  # 3-fold for speed
                                scoring='neg_mean_squared_error')
    return -cv_scores.mean()

study_etr = optuna.create_study(direction='minimize', study_name='etr_tuning')
study_etr.optimize(objective_etr, n_trials=30, show_progress_bar=True)

print("Best ExtraTrees params:", study_etr.best_params)
print("Best CV MSE:", study_etr.best_value)

[I 2026-04-07 02:01:29,078] A new study created in memory with name: etr_tuning


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-07 02:02:33,117] Trial 0 finished with value: 2.317564036319873 and parameters: {'n_estimators': 50, 'max_depth': 38, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 2.317564036319873.
[I 2026-04-07 02:11:22,152] Trial 1 finished with value: 1.6076316605972034 and parameters: {'n_estimators': 100, 'max_depth': 21, 'min_samples_split': 4, 'min_samples_leaf': 10, 'max_features': 0.7}. Best is trial 1 with value: 1.6076316605972034.
[I 2026-04-07 02:35:45,741] Trial 2 finished with value: 1.5668569764174898 and parameters: {'n_estimators': 250, 'max_depth': 24, 'min_samples_split': 14, 'min_samples_leaf': 7, 'max_features': 0.7}. Best is trial 2 with value: 1.5668569764174898.
[I 2026-04-07 02:42:35,917] Trial 3 finished with value: 1.9457978268250093 and parameters: {'n_estimators': 300, 'max_depth': 43, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 2 with value: 1.5668569764174898.
[I

KeyboardInterrupt: 

In [ ]:
# Best ExtraTrees
best_etr = ExtraTreesRegressor(**study_etr.best_params, random_state=42, n_jobs=-1)
best_etr.fit(X_train_scaled, y_train)
y_pred_etr_tuned = best_etr.predict(X_test_scaled)

r2_etr_tuned = r2_score(y_test, y_pred_etr_tuned)
mae_etr_tuned = mean_absolute_error(y_test, y_pred_etr_tuned)
rmse_etr_tuned = np.sqrt(mean_squared_error(y_test, y_pred_etr_tuned))


print("Tuned Extra Trees")
print(f"R²:  {r2_etr_tuned:.4f}")
print(f"MAE: {mae_etr_tuned:.4f}")
print(f"RMSE:{rmse_etr_tuned:.4f}")

In [ ]:
comparison_tuned = pd.DataFrame({
    "Model": ["Linear Regression", "Extra Trees (default)", "Extra Trees (tuned)"],
    "R²": [r2_lr, r2_ridge, r2_etr, r2_etr_tuned],
    "MAE": [mae_lr, mae_ridge, mae_etr, mae_etr_tuned],
    "RMSE": [rmse_lr, rmse_ridge, rmse_etr, rmse_etr_tuned]
})
print(comparison_tuned.round(4))